# Prepare MAUDE summaries for Streamlit

Your device summary already includes `PERCENT_OF_COMPANY_YEAR`. This notebook verifies and recalculates it, preserving raw counts and adding a fixed 2021 index. No random downsampling or arbitrary rescaling is needed.

The default chart answers: **What share of a company's device records that year concern this device type?** It does not estimate risk or incidence. The source unit is device records, not confirmed unique adverse-event reports.

Run cells in order. Upload one device-type summary CSV and the company/year summary CSV. The two previously supplied device-type files are identical, so either is sufficient. Your original notebook is unchanged.


In [ ]:
from google.colab import files
uploaded = files.upload()  # Select both summary CSVs together.


## 1. Identify files by their columns

This accepts renamed files and identical duplicate uploads. It stops if conflicting copies of a summary are uploaded.


In [ ]:
import io
import hashlib
import pandas as pd
from pathlib import Path

candidates = {'devices': [], 'companies': []}
for name, content in uploaded.items():
    if not name.lower().endswith('.csv'):
        continue
    columns = set(pd.read_csv(io.BytesIO(content), nrows=0, encoding='utf-8-sig').columns)
    if {'YEAR', 'ANALYSIS_COMPANY', 'DEVICE_RECORD_COUNT'}.issubset(columns):
        kind = 'devices' if 'GENERIC_DEVICE_TYPE' in columns else 'companies'
        candidates[kind].append((name, content))
for kind, entries in candidates.items():
    if not entries:
        raise ValueError(f'Upload the {kind} summary CSV, then rerun the upload cell.')
    if len({hashlib.sha256(content).hexdigest() for _, content in entries}) > 1:
        raise ValueError(f'Multiple different {kind} files were uploaded. Choose one version.')
    print(kind, '→', entries[0][0])


## 2. Validate and calculate

The following reusable function checks company/year totals and the supplied rounded percentages. It labels NA/N/A categories as `Unknown / not specified` and combines matching keys after that cleaning, retaining all counts. It does not combine clinically similar names or treat absent rows as zero.

- Share = category count / company-year total × 100.
- Index = category count / the same company's same category count in 2021 × 100.
- Missing or zero denominators yield missing values.


In [ ]:
"""Reproducible preparation for the supplied MAUDE summary files."""
from pathlib import Path
import argparse
import json
import numpy as np
import pandas as pd

KEY = ['YEAR', 'ANALYSIS_COMPANY']
DEVICE = 'GENERIC_DEVICE_TYPE'
COUNT = 'DEVICE_RECORD_COUNT'

def prepare(device_path, company_path):
    # Preserve literal NA/N/A values until we intentionally label them.
    d = pd.read_csv(device_path, encoding='utf-8-sig', keep_default_na=False)
    c = pd.read_csv(company_path, encoding='utf-8-sig', keep_default_na=False)
    for frame, required in [(d, KEY + [DEVICE, COUNT]), (c, KEY + [COUNT])]:
        missing = set(required) - set(frame.columns)
        if missing:
            raise ValueError(f'Missing required columns: {sorted(missing)}')
        for col in ['YEAR', COUNT]:
            frame[col] = pd.to_numeric(frame[col], errors='raise')
            if not np.isfinite(frame[col]).all() or (frame[col] % 1 != 0).any():
                raise ValueError(f'{col} must contain finite integers.')
            frame[col] = frame[col].astype('int64')
        if (frame[COUNT] < 0).any():
            raise ValueError('Counts cannot be negative.')
        frame['ANALYSIS_COMPANY'] = frame['ANALYSIS_COMPANY'].str.strip()
        if frame['ANALYSIS_COMPANY'].eq('').any():
            raise ValueError('Company names cannot be empty.')
        if not frame.YEAR.isin(range(2021, 2026)).all():
            raise ValueError('This dashboard expects years 2021–2025.')
    if c.duplicated(KEY).any():
        raise ValueError('Company/year totals must be unique.')
    if d.duplicated(KEY + [DEVICE]).any():
        raise ValueError('Duplicate source company/year/device keys; inspect before combining.')
    joined = d.merge(c[KEY + [COUNT]].rename(columns={COUNT:'COMPANY_YEAR_TOTAL'}),
                     on=KEY, how='left', validate='many_to_one')
    if joined.COMPANY_YEAR_TOTAL.isna().any():
        raise ValueError('Some device rows have no company/year denominator.')
    expected = joined[COUNT].div(joined.COMPANY_YEAR_TOTAL.replace(0, np.nan)).mul(100)
    original_error = None
    if 'PERCENT_OF_COMPANY_YEAR' in joined:
        supplied = pd.to_numeric(joined.PERCENT_OF_COMPANY_YEAR, errors='raise')
        original_error = float((supplied - expected).abs().max())
        if not np.allclose(supplied, expected, atol=0.000051, rtol=0, equal_nan=True):
            raise ValueError('Source percentages do not match company/year denominators.')
    d[DEVICE] = d[DEVICE].str.strip()
    unknown = d[DEVICE].str.upper().isin(['', 'NA', 'N/A', 'NAN', 'NULL'])
    unknown_rows, unknown_records = int(unknown.sum()), int(d.loc[unknown, COUNT].sum())
    d.loc[unknown, DEVICE] = 'Unknown / not specified'
    # Aggregate only exact matching labels after whitespace/missing-label cleaning.
    grouped = d.groupby(KEY + [DEVICE], as_index=False)[COUNT].sum()
    totals = grouped.groupby(KEY)[COUNT].sum().sort_index()
    control = c.set_index(KEY)[COUNT].sort_index()
    if not totals.index.equals(control.index) or not totals.equals(control):
        raise ValueError('Device totals do not reconcile with company/year totals.')
    out = grouped.merge(c[KEY + [COUNT]].rename(columns={COUNT:'COMPANY_YEAR_TOTAL'}),
                        on=KEY, validate='many_to_one')
    out['PERCENT_OF_COMPANY_YEAR'] = out[COUNT].div(out.COMPANY_YEAR_TOTAL.replace(0, np.nan)).mul(100)
    baseline = out.loc[out.YEAR.eq(2021), ['ANALYSIS_COMPANY', DEVICE, COUNT]].rename(columns={COUNT:'BASELINE_2021_COUNT'})
    out = out.merge(baseline, on=['ANALYSIS_COMPANY', DEVICE], how='left', validate='many_to_one')
    out['INDEX_2021_100'] = out[COUNT].div(out.BASELINE_2021_COUNT.replace(0, np.nan)).mul(100)
    if 'SOURCE_URL' in c:
        out = out.merge(c[KEY + ['SOURCE_URL']], on=KEY, validate='many_to_one')
    summary = {
        'source_device_rows': len(d), 'prepared_device_rows': len(out),
        'company_year_rows': len(c), 'companies': int(c.ANALYSIS_COMPANY.nunique()),
        'years': sorted(c.YEAR.unique().tolist()), 'total_device_records': int(c[COUNT].sum()),
        'unknown_label_source_rows': unknown_rows, 'unknown_label_device_records': unknown_records,
        'max_source_percentage_rounding_difference': original_error,
        'all_company_year_totals_reconcile': True,
        'year_coverage': 'Year labels verified; extraction date, year basis, and full-year coverage unverified.',
        'unit': 'Device records; unique reports/events cannot be recovered from these summaries.'
    }
    return out.sort_values(KEY + [DEVICE]).reset_index(drop=True), c, summary


In [ ]:
device_trends, company_year, validation = prepare(
    io.BytesIO(candidates['devices'][0][1]),
    io.BytesIO(candidates['companies'][0][1]),
)
print(json.dumps(validation, indent=2))
display(device_trends.head())


## 3. Preview the five original trends with normalization

The original notebook used record counts. Here, the same five company/device combinations use percentages. Change `selected_pairs` to choose others. Gaps mean no row is available. Counts and shares are both useful, but they answer different questions.


In [ ]:
import matplotlib.pyplot as plt

selected_pairs = [
    ('BD', 'PUMP, INFUSION'),
    ('Medtronic', 'AUTOMATED INSULIN DOSING DEVICE SYSTEM, SINGLE HORMONAL CONTROL'),
    ('Medtronic', 'ARTIFICIAL PANCREAS DEVICE SYSTEM, SINGLE HORMONAL CONTROL'),
    ('Abbott', 'FLASH GLUCOSE MONITORING SYSTEM'),
    ('Medtronic', 'PUMP, INFUSION, INSULIN, TO BE USED WITH INVASIVE GLUCOSE SENSOR'),
]
fig, ax = plt.subplots(figsize=(12, 7))
for company, device in selected_pairs:
    part = device_trends.loc[
        device_trends.ANALYSIS_COMPANY.eq(company) &
        device_trends.GENERIC_DEVICE_TYPE.eq(device)
    ].set_index('YEAR').reindex(range(2021, 2026))
    ax.plot(part.index, part.PERCENT_OF_COMPANY_YEAR, marker='o',
            label=f'{company}: {device}')
ax.set(title='Device categories as a share of company-year records',
       xlabel='Year', ylabel='Share of company-year device records (%)')
ax.set_xticks(range(2021, 2026))
ax.set_ylim(bottom=0)
ax.grid(alpha=0.2)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.14), fontsize=8)
fig.tight_layout()
plt.show()


## 4. Export the Streamlit-ready data

Extract the downloaded ZIP and place `device_trends.csv`, `company_year.csv`, and `validation.json` in the dashboard's `data` folder. Keep the existing narrative and outcome snapshot files there. The delivered dashboard already contains these prepared tables, so this step is only needed when reproducing or updating them.


In [ ]:
import zipfile
output_dir = Path('maude_prepared')
output_dir.mkdir(exist_ok=True)
device_trends.to_csv(output_dir / 'device_trends.csv', index=False)
company_year.to_csv(output_dir / 'company_year.csv', index=False)
(output_dir / 'validation.json').write_text(json.dumps(validation, indent=2))
with zipfile.ZipFile('maude_prepared_data.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for filename in ['device_trends.csv', 'company_year.csv', 'validation.json']:
        archive.write(output_dir / filename, filename)
files.download('maude_prepared_data.zip')


## Interpretation

All 50 supplied company/year totals reconcile. The summaries do not establish a company safety ranking, the number of unique adverse events, device-use exposure, causes of spikes, or full-year coverage. The date basis of YEAR and extraction date remain unverified. Device naming differences may fragment related categories; product codes or a reviewed mapping would be needed to harmonize them.

FDA guidance: https://open.fda.gov/data/maude/
